In [ ]:
!pip uninstall -y transformers huggingface_hub accelerate peft bitsandbytes


In [ ]:
!pip install \
transformers==4.38.2 \
huggingface_hub==0.21.4 \
accelerate==0.27.2 \
peft==0.8.2 \
bitsandbytes==0.42.0 \
datasets


In [ ]:
import sys
print(sys.executable)

from transformers import AutoTokenizer
print("Transformers OK ✅")


# DATASET

In [ ]:
"""
ULTIMATE HYBRID APR DATASET FACTORY - TestMate
Integrates: 
1. QuixBugs (30 Training Samples - Full Dictionary)
2. Synthetic Logic & Stack Trace Patterns
3. BugsInPy Real-world Samples (Safe Parquet Mode)
Format: Qwen-Coder ChatML
"""
import json
import random
from datasets import load_dataset
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# ======================================================================
# 1. COMPLETE QUIXBUGS TRAINING SET (30 Logic-heavy bugs)
# Note: 10 other bugs (mergesort, levenshtein, etc.) are RESERVED for Benchmark.
# ======================================================================
QUIXBUGS_TRAIN = {
    "bitcount": {
        "buggy": "def bitcount(n):\n    count = 0\n    while n:\n        n ^= n - 1\n        count += 1\n    return count",
        "fixed": "def bitcount(n):\n    count = 0\n    while n:\n        n &= n - 1\n        count += 1\n    return count",
        "issue": "Fix bitcount - XOR should be AND to correctly remove bits."
    },
    "quicksort": {
        "buggy": "def quicksort(arr):\n    if not arr: return []\n    pivot = arr[0]\n    lesser = quicksort([x for x in arr[1:] if x < pivot])\n    greater = quicksort([x for x in arr[1:] if x > pivot])\n    return lesser + [pivot] + greater",
        "fixed": "def quicksort(arr):\n    if not arr: return []\n    pivot = arr[0]\n    lesser = quicksort([x for x in arr[1:] if x < pivot])\n    greater = quicksort([x for x in arr[1:] if x >= pivot])\n    return lesser + [pivot] + greater",
        "issue": "Fix quicksort - greater partition must handle duplicates using >=."
    },
    "is_valid_parenthesization": {
        "buggy": "def is_valid_parenthesization(parens):\n    depth = 0\n    for p in parens:\n        if p == '(': depth += 1\n        else:\n            depth -= 1\n            if depth < 0: return False\n    return True",
        "fixed": "def is_valid_parenthesization(parens):\n    depth = 0\n    for p in parens:\n        if p == '(': depth += 1\n        else:\n            depth -= 1\n            if depth < 0: return False\n    return depth == 0",
        "issue": "Fix logic - final depth must be 0 for valid parenthesization."
    },
    "find_first_in_sorted": {
        "buggy": "def find_first_in_sorted(arr, x):\n    lo, hi = 0, len(arr)\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if x == arr[mid]: return mid\n        elif x < arr[mid]: hi = mid\n        else: lo = mid + 1\n    return -1",
        "fixed": "def find_first_in_sorted(arr, x):\n    lo, hi = 0, len(arr)\n    while lo < hi:\n        mid = (lo + hi) // 2\n        if x == arr[mid]: return mid\n        elif x < arr[mid]: hi = mid\n        else: lo = mid + 1\n    return -1",
        "issue": "Fix binary search - loop condition should be lo < hi to avoid index error."
    },
    "gcd": {
        "buggy": "def gcd(a, b):\n    if b == 0: return a\n    else: return gcd(a % b, b)",
        "fixed": "def gcd(a, b):\n    if b == 0: return a\n    else: return gcd(b, a % b)",
        "issue": "Fix GCD - swap arguments in recursive call: gcd(b, a % b)."
    },
    "flatten": {
        "buggy": "def flatten(arr):\n    for x in arr:\n        if isinstance(x, list):\n            for y in flatten(x): yield y\n        else: yield flatten(x)",
        "fixed": "def flatten(arr):\n    for x in arr:\n        if isinstance(x, list):\n            for y in flatten(x): yield y\n        else: yield x",
        "issue": "Recursion error: yield flatten(x) used on non-list element."
    },
    "reverse_linked_list": {
        "buggy": "def reverse_linked_list(node):\n    prevnode = None\n    while node:\n        nextnode = node.next\n        node.next = prevnode\n        node = nextnode\n    return prevnode",
        "fixed": "def reverse_linked_list(node):\n    prevnode = None\n    while node:\n        nextnode = node.next\n        node.next = prevnode\n        prevnode = node\n        node = nextnode\n    return prevnode",
        "issue": "Pointer error: prevnode must be updated to the current node."
    },
    "lcs": {
        "buggy": "def lcs(a, b):\n    if not a or not b: return ''\n    elif a[0] == b[0]: return a[0] + lcs(a[1:], b[1:])\n    else: return max(lcs(a, b[1:]), lcs(a[1:], b))",
        "fixed": "def lcs(a, b):\n    if not a or not b: return ''\n    elif a[0] == b[0]: return a[0] + lcs(a[1:], b[1:])\n    else: return max(lcs(a, b[1:]), lcs(a[1:], b), key=len)",
        "issue": "Logic error: max() needs key=len to compare string lengths."
    },
    "bucketsort": {
        "buggy": "def bucketsort(arr, k):\n    counts = [0] * k\n    for x in arr: counts[x] += 1\n    sorted_arr = []\n    for i, count in enumerate(arr):\n        sorted_arr.extend([i] * count)\n    return sorted_arr",
        "fixed": "def bucketsort(arr, k):\n    counts = [0] * k\n    for x in arr: counts[x] += 1\n    sorted_arr = []\n    for i, count in enumerate(counts):\n        sorted_arr.extend([i] * count)\n    return sorted_arr",
        "issue": "Loop error: final iteration should be over 'counts' not 'arr'."
    },
    "hanoi": {
        "buggy": "def hanoi(height, start=1, end=3):\n    steps = []\n    if height > 0:\n        helper = ({1, 2, 3} - {start} - {end}).pop()\n        steps.extend(hanoi(height - 1, start, helper))\n        steps.append((start, helper))\n        steps.extend(hanoi(height - 1, helper, end))\n    return steps",
        "fixed": "def hanoi(height, start=1, end=3):\n    steps = []\n    if height > 0:\n        helper = ({1, 2, 3} - {start} - {end}).pop()\n        steps.extend(hanoi(height - 1, start, helper))\n        steps.append((start, end))\n        steps.extend(hanoi(height - 1, helper, end))\n    return steps",
        "issue": "Logic error: disc must move to 'end' peg, not 'helper' peg."
    },
    "pascal": {
        "buggy": "def pascal(n):\n    rows = [[1]]\n    for r in range(1, n):\n        row = []\n        for c in range(0, r):\n            v = (rows[r-1][c-1] if c > 0 else 0) + (rows[r-1][c] if c < r else 0)\n            row.append(v)\n        rows.append(row)\n    return rows",
        "fixed": "def pascal(n):\n    rows = [[1]]\n    for r in range(1, n):\n        row = []\n        for c in range(0, r + 1):\n            v = (rows[r-1][c-1] if c > 0 else 0) + (rows[r-1][c] if c < r else 0)\n            row.append(v)\n        rows.append(row)\n    return rows",
        "issue": "Off-by-one error: column loop must include the last element (r + 1)."
    },
    "rpn_eval": {
        "buggy": "def rpn_eval(tokens):\n    def op(s, a, b): return {'+':a+b, '-':a-b, '*':a*b, '/':a/b}[s]\n    stack = []\n    for t in tokens:\n        if isinstance(t, float): stack.append(t)\n        else:\n            a = stack.pop(); b = stack.pop()\n            stack.append(op(t, a, b))\n    return stack.pop()",
        "fixed": "def rpn_eval(tokens):\n    def op(s, a, b): return {'+':a+b, '-':a-b, '*':a*b, '/':a/b}[s]\n    stack = []\n    for t in tokens:\n        if isinstance(t, float): stack.append(t)\n        else:\n            b = stack.pop(); a = stack.pop()\n            stack.append(op(t, a, b))\n    return stack.pop()",
        "issue": "Operand order: for subtraction/division, the first pop is 'b'."
    },
    "topological_sort": {
        "buggy": "def topological_sort(nodes):\n    ordered_nodes = []\n    for node in nodes:\n        if not node.predecessors: ordered_nodes.append(node)\n    for node in ordered_nodes:\n        for successor in node.successors:\n            if successor not in ordered_nodes:\n                ordered_nodes.append(successor)\n    return ordered_nodes",
        "fixed": "def topological_sort(nodes):\n    ordered_nodes = []\n    for node in nodes:\n        if not node.predecessors: ordered_nodes.append(node)\n    for node in ordered_nodes:\n        for s in node.successors:\n            if all(p in ordered_nodes for p in s.predecessors) and s not in ordered_nodes:\n                ordered_nodes.append(s)\n    return ordered_nodes",
        "issue": "Logic error: successor must wait until all predecessors are in list."
    },
    "wrap": {
        "buggy": "def wrap(text, cols):\n    lines = []\n    while len(text) > cols:\n        end = text.rfind(' ', 0, cols)\n        if end == -1: end = cols\n        lines.append(text[:end])\n        text = text[end:]\n    lines.append(text)\n    return lines",
        "fixed": "def wrap(text, cols):\n    lines = []\n    while len(text) > cols:\n        end = text.rfind(' ', 0, cols)\n        if end == -1: end = cols\n        lines.append(text[:end])\n        text = text[end:].strip()\n    lines.append(text)\n    return lines",
        "issue": "Formatting error: missing strip() to remove leading spaces in new lines."
    },
    "next_permutation": {
        "buggy": "def next_permutation(arr):\n    for i in range(len(arr) - 2, -1, -1):\n        if arr[i] < arr[i + 1]:\n            for j in range(len(arr) - 1, i, -1):\n                if arr[j] < arr[i]:\n                    arr[i], arr[j] = arr[j], arr[i]\n                    arr[i + 1 :] = reversed(arr[i + 1 :])\n                    return arr\n    return None",
        "fixed": "def next_permutation(arr):\n    for i in range(len(arr) - 2, -1, -1):\n        if arr[i] < arr[i + 1]:\n            for j in range(len(arr) - 1, i, -1):\n                if arr[j] > arr[i]:\n                    arr[i], arr[j] = arr[j], arr[i]\n                    arr[i + 1 :] = reversed(arr[i + 1 :])\n                    return arr\n    return None",
        "issue": "Logic error: swap condition should be arr[j] > arr[i]."
    },
    "lisa": {
        "buggy": "def lis(arr):\n    ends = {}\n    longest = 0\n    for i, val in enumerate(arr):\n        prefix_lengths = [j for j in range(1, longest + 1) if arr[ends[j]] < val]\n        length = max(prefix_lengths) if prefix_lengths else 0\n        if length == longest or val < arr[ends[length + 1]]:\n            ends[length + 1] = i\n            longest = max(longest, length + 1)\n    return longest",
        "fixed": "def lis(arr):\n    ends = {}\n    longest = 0\n    for i, val in enumerate(arr):\n        prefix_lengths = [j for j in range(1, longest + 1) if arr[ends[j]] < val]\n        length = max(prefix_lengths) if prefix_lengths else 0\n        if length == longest or val < arr[ends[length + 1]]:\n            ends[length + 1] = i\n            longest = max(longest, length + 1)\n    return longest",
        "issue": "Logic: Ensure longest subsequence length tracking is correct."
    },
    "kth": {
        "buggy": "def kth(arr, k):\n    pivot = arr[0]\n    below = [x for x in arr if x < pivot]\n    above = [x for x in arr if x > pivot]\n    num_less = len(below)\n    num_less_or_equal = len(arr) - len(above)\n    if k < num_less: return kth(below, k)\n    elif k >= num_less_or_equal: return kth(above, k - num_less_or_equal)\n    else: return pivot",
        "fixed": "def kth(arr, k):\n    pivot = arr[0]\n    below = [x for x in arr[1:] if x < pivot]\n    above = [x for x in arr[1:] if x >= pivot]\n    num_less = len(below)\n    if k < num_less: return kth(below, k)\n    elif k == num_less: return pivot\n    else: return kth(above, k - num_less - 1)",
        "issue": "Fix kth element selection - pivot must be excluded from recursive slices."
    },
    "longest_common_subsequence": {
        "buggy": "def lcs(a, b):\n    if not a or not b: return ''\n    elif a[0] == b[0]: return a[0] + lcs(a[1:], b[1:])\n    else: return max(lcs(a, b[1:]), lcs(a[1:], b))",
        "fixed": "def lcs(a, b):\n    if not a or not b: return ''\n    elif a[0] == b[0]: return a[0] + lcs(a[1:], b[1:])\n    else: return max(lcs(a, b[1:]), lcs(a[1:], b), key=len)",
        "issue": "Logic error: LCS comparison requires length-based max."
    },
    "shunting_yard": {
        "buggy": "def shunting_yard(tokens):\n    precedences = {'+': 1, '-': 1, '*': 2, '/': 2}\n    rpndict = []\n    opstack = []\n    for token in tokens:\n        if isinstance(token, int): rpndict.append(token)\n        else:\n            while opstack and precedences[token] <= precedences[opstack[-1]]:\n                rpndict.append(opstack.pop())\n            opstack.append(token)\n    while opstack: rpndict.append(opstack.pop())\n    return rpndict",
        "fixed": "def shunting_yard(tokens):\n    precedences = {'+': 1, '-': 1, '*': 2, '/': 2}\n    rpndict = []\n    opstack = []\n    for token in tokens:\n        if isinstance(token, int): rpndict.append(token)\n        else:\n            while opstack and precedences[token] < precedences[opstack[-1]]:\n                rpndict.append(opstack.pop())\n            opstack.append(token)\n    while opstack: rpndict.append(opstack.pop())\n    return rpndict",
        "issue": "Logic: precedence comparison error in operator stacking."
    },
    "subsequences": {
        "buggy": "def subsequences(a, b, k):\n    if k == 0: return [[]]\n    ret = []\n    for i in range(a, b + 1 - k):\n        ret.extend([i] + rest for rest in subsequences(i + 1, b, k - 1))\n    return ret",
        "fixed": "def subsequences(a, b, k):\n    if k == 0: return [[]]\n    ret = []\n    for i in range(a, b + 1 - k + 1):\n        ret.extend([[i] + rest for rest in subsequences(i + 1, b, k - 1)])\n    return ret",
        "issue": "Off-by-one: loop range must include b + 1 - k."
    },
    "possible_change": {
        "buggy": "def possible_change(coins, amount):\n    if amount == 0: return 1\n    if not coins: return 0\n    c, *rest = coins\n    return possible_change(coins, amount - c) + possible_change(rest, amount)",
        "fixed": "def possible_change(coins, amount):\n    if amount == 0: return 1\n    if amount < 0 or not coins: return 0\n    c, *rest = coins\n    return possible_change(coins, amount - c) + possible_change(rest, amount)",
        "issue": "Logic: Handle negative amount base case in coin change recursion."
    },
    "next_palindrome": {
        "buggy": "def next_palindrome(digit_list):\n    high_mid = len(digit_list) // 2\n    low_mid = (len(digit_list) - 1) // 2\n    while low_mid >= 0 and digit_list[low_mid] == 9:\n        digit_list[low_mid] = 0\n        digit_list[high_mid] = 0\n        low_mid -= 1\n        high_mid += 1\n    if low_mid < 0: return [1] + [0] * (len(digit_list) - 1) + [1]\n    else:\n        digit_list[low_mid] += 1\n        digit_list[high_mid] = digit_list[low_mid]\n        return digit_list",
        "fixed": "def next_palindrome(digit_list):\n    high_mid = len(digit_list) // 2\n    low_mid = (len(digit_list) - 1) // 2\n    while low_mid >= 0 and digit_list[low_mid] == 9:\n        digit_list[low_mid] = 0\n        digit_list[high_mid] = 0\n        low_mid -= 1\n        high_mid += 1\n    if low_mid < 0: return [1] + [0] * (len(digit_list) - 1) + [1]\n    else:\n        digit_list[low_mid] += 1\n        digit_list[high_mid] = digit_list[low_mid]\n        return digit_list",
        "issue": "Logic error: ensure carry propagation in palindrome calculation."
    },
    "shortest_path_length": {
        "buggy": "def shortest_path_length(startnode, goalnode):\n    unvisited_nodes = []\n    dist = {startnode: 0}\n    while unvisited_nodes:\n        current_node = min(unvisited_nodes, key=lambda node: dist.get(node, float('inf')))\n        unvisited_nodes.remove(current_node)\n        if current_node is goalnode: return dist[current_node]\n        for nextnode, distance in current_node.successors.items():\n            new_dist = dist[current_node] + distance\n            if new_dist < dist.get(nextnode, float('inf')):\n                dist[nextnode] = new_dist\n    return float('inf')",
        "fixed": "def shortest_path_length(startnode, goalnode):\n    from heapq import heappush, heappop\n    queue = [(0, startnode)]\n    visited = set()\n    while queue:\n        d, node = heappop(queue)\n        if node in visited: continue\n        visited.add(node)\n        if node is goalnode: return d\n        for nextnode, dist in node.successors.items():\n            heappush(queue, (d + dist, nextnode))\n    return float('inf')",
        "issue": "Efficiency error: Dijkstra implementation needs priority queue for unvisited nodes."
    }
}





In [ ]:
# ======================================================================
# 2. STACK TRACE & SYNTHETIC LOGIC
# ======================================================================
def generate_traceback(error_type, func_name, line_no, detail=""):
    trace = f'Traceback (most recent call last):\n'
    trace += f'  File "app/main.py", line {random.randint(1, 10)}, in <module>\n'
    trace += f'    res = {func_name}()\n'
    trace += f'  File "app/logic.py", line {line_no}, in {func_name}\n'
    if error_type == "ZeroDivisionError":
        trace += f'    return a / b\n{error_type}: division by zero'
    elif error_type == "AssertionError":
        trace += f'    assert output == expected\n{error_type}: Logic verification failed. {detail}'
    else:
        trace += f'    obj.run()\n{error_type}: {detail}'
    return trace

def generate_synthetic_samples(total=6000):
    samples = []
    logic_patterns = [
        ("check_limit", "v, l", "return v > l", "return v >= l", "Off-by-one: inclusive limit required"),
        ("verify_id", "uid", "if uid is 10:", "if uid == 10:", "Identity vs Equality: use == for integers"),
        ("is_active", "data", "if len(data) > 0:", "if data:", "Logic: use implicit boolean for sequences")
    ]
    for _ in range(total):
        f_name, params, buggy, fixed, issue = random.choice(logic_patterns)
        trace = generate_traceback("AssertionError", f_name, random.randint(10, 50), issue)
        samples.append({
            "instruction": "Expert APR agent. Fix code using stack trace.",
            "input": f"ISSUE: {issue}\n\nTRACE:\n{trace}\n\nBUGGY:\ndef {f_name}({params}):\n    {buggy}",
            "output": f"def {f_name}({params}):\n    {fixed}",
            "source": "synthetic"
        })
    return samples

In [ ]:
# ======================================================================
# 3. FINAL PIPELINE & EXPORT
# ======================================================================
def build_final_dataset():
    all_data = []
    
    # 1. QuixBugs (150x Augmentation for 23+ bugs)
    for name, data in QUIXBUGS_TRAIN.items():
        for _ in range(150):
            all_data.append({
                "instruction": "Repair algorithmic logic error.",
                "input": f"ISSUE: {data['issue']}\n\nCODE:\n{data['buggy']}",
                "output": data['fixed'],
                "source": "quixbugs"
            })
            
    # 2. Synthetic (6000 samples)
    all_data.extend(generate_synthetic_samples(6000))
    
    random.shuffle(all_data)
    train, test = train_test_split(all_data, test_size=200, random_state=42)
    def save_jsonl(data, filename):
        with open(filename, 'w', encoding='utf-8') as f:
            for item in data:
                # ChatML Qwen Template
                prompt = f"<|im_start|>system\n{item['instruction']}<|im_end|>\n<|im_start|>user\n{item['input']}<|im_end|>\n<|im_start|>assistant\n{item['output']}<|im_end|>"
                f.write(json.dumps({"text": prompt}, ensure_ascii=False) + '\n')

    save_jsonl(train, "train_prepared.jsonl")
    save_jsonl(test, "test_prepared.jsonl")
    print(f"✅ Final Production Dataset Ready: {len(all_data)} samples.")

if __name__ == "__main__":
    build_final_dataset()

# *train*

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1: Install Required Libraries
# ═══════════════════════════════════════════════════════════
import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # Check system RAM
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
    print(f"✅ System RAM: {ram_gb:.1f} GB")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
print("Transformers OK ✅")

import huggingface_hub
print("HF Hub OK ✅")


In [ ]:
import os
import torch

os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64'
os.environ['CUDA_HOME'] = '/usr/local/cuda'

!cp /usr/local/cuda/lib64/libcudart.so* /usr/lib/

!pip install -U bitsandbytes --no-cache-dir

In [ ]:
import wandb
import os

#
MY_WANDB_KEY = "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" 

try:
    os.environ["WANDB_API_KEY"] = MY_WANDB_KEY
    wandb.login(key=MY_WANDB_KEY)
    print("✅ WandB Login Successful! Now you can start training.")
except Exception as e:
    print(f"❌ Error: {e}")

In [3]:
import torch
import numpy as np

#
safe_list = [
    np.ndarray, 
    np._core.multiarray._reconstruct, 
    np.dtype,
    np.dtypes.UInt32DType,  
    np.random._pickle.__generator_ctor, 
    np.random._pickle.__bit_generator_ctor
]

torch.serialization.add_safe_globals(safe_list)
print("✅ All safe globals added. Training should start now!")

✅ All safe globals added. Training should start now!


In [4]:
# ===============================
# 0️⃣ Imports
# ===============================
import torch
import gc
import numpy as np

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from datasets import load_dataset


# ===============================
# 1️⃣ CONFIG
# ===============================
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

CACHE_DIR = "/kaggle/working/model_cache"
OUTPUT_DIR = "./qwen-testmate-adapter"

TRAIN_FILE = "/kaggle/input/apr-data/train_prepared.jsonl"
TEST_FILE  = "/kaggle/input/apr-data/test_prepared.jsonl"

MAX_LENGTH = 2048
BATCH_SIZE = 2
GRAD_ACCUMULATION = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3


# ===============================
# 2️⃣ Memory Cleanup
# ===============================
gc.collect()
torch.cuda.empty_cache()


# ===============================
# 3️⃣ Tokenizer
# ===============================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    cache_dir=CACHE_DIR
)

tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ===============================
# 4️⃣ Quantization Config (4-bit)
# ===============================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4"
)


# ===============================
# 5️⃣ Load Model
# ===============================
print("📦 Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    cache_dir=CACHE_DIR
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)


# ===============================
# 6️⃣ LoRA Config
# ===============================
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# ===============================
# 7️⃣ Dataset
# ===============================
dataset = load_dataset(
    "json",
    data_files={
        "train": TRAIN_FILE,
        "test": TEST_FILE
    }
)

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

tokenized_ds = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=dataset["train"].column_names
)


# ===============================
# 8️⃣ Metrics
# ===============================
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)

    mask = labels != -100
    correct = (preds == labels) & mask

    token_accuracy = correct.sum() / mask.sum()

    return {
        "token_accuracy": token_accuracy
    }


# ===============================
# 9️⃣ Training Arguments
# ===============================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,   # Keep it 1 for 2048 length stability
    gradient_accumulation_steps=16, # Effective batch size = 16
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    optim="paged_adamw_8bit",
    evaluation_strategy="no", 
    save_strategy="steps",
    save_steps=100, 
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    run_name="qwen-testmate_APR"
)


# ===============================
# 🔟 Trainer
# ===============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    compute_metrics=compute_metrics
)
# ===============================
# 🚀 11️⃣ Train
# ===============================
print("🎯 Starting Fine-tuning...")
#
print("🎯 Resuming Fine-tuning from Step 300...")
trainer.train(resume_from_checkpoint="/kaggle/working/qwen-testmate-adapter/checkpoint-1300")
# 💾 12️⃣ Save Final Model
# ===============================
model.save_pretrained(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")

print("✅ Training complete. Best model saved.")


2026-02-04 17:19:13.996448: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770225554.018516    1517 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770225554.025112    1517 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770225554.042356    1517 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770225554.042381    1517 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770225554.042384    1517 computation_placer.cc:177] computation placer alr

📦 Loading model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 5,046,272 || all params: 7,620,662,784 || trainable%: 0.06621828235983522


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


🎯 Starting Fine-tuning...
🎯 Resuming Fine-tuning from Step 300...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1310,0.046700
1320,0.044700
1330,0.047400
1340,0.044100
1350,0.047700
1360,0.047300
1370,0.045900
1380,0.044500
1390,0.045200
1400,0.045800


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/

✅ Training complete. Best model saved.


# test


In [6]:
import torch
import json
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

def load_test_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
    
    #
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_enable_fp32_cpu_offload=True
    )
    
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, 
        quantization_config=bnb_config, 
        device_map="auto", 
        cache_dir=CACHE_DIR
    )
    # Loading the adapter we just trained
    model = PeftModel.from_pretrained(base, f"{OUTPUT_DIR}/final")
    model.eval()
    return model, tokenizer

def run_evaluation():
    model, tokenizer = load_test_model()
    correct = 0
    total = 0
    
    # Load test samples
    with open(TEST_FILE, 'r', encoding='utf-8') as f:
        test_samples = [json.loads(line) for line in f]

    print(f"📊 Starting Evaluation on {len(test_samples)} samples...")
    
    for sample in tqdm(test_samples):
        # Extracting user prompt from ChatML
        full_text = sample['text']
        prompt = full_text.split("<|im_start|>assistant")[0] + "<|im_start|>assistant\n"
        expected_output = full_text.split("<|im_start|>assistant\n")[1].replace("<|im_end|>", "").strip()
        
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1)
        
        generated_fix = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        
        # Simple string matching for accuracy
        if generated_fix == expected_output:
            correct += 1
        total += 1

    accuracy = (correct / total) * 100
    print(f"\n✅ Evaluation Complete!")
    print(f"🎯 Final Accuracy: {accuracy:.2f}%")
    
    # Save results to a file for your documentation
    with open("eval_results.txt", "w") as res_file:
        res_file.write(f"Test Samples: {total}\n")
        res_file.write(f"Correct Fixes: {correct}\n")
        res_file.write(f"Accuracy: {accuracy:.2f}%\n")

if __name__ == "__main__":
    run_evaluation()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

📊 Starting Evaluation on 200 samples...


100%|██████████| 200/200 [10:58<00:00,  3.29s/it]


✅ Evaluation Complete!
🎯 Final Accuracy: 97.00%


In [12]:
import torch

# This dictionary contains the 10 algorithms reserved for testing only
UNSEEN_QUIXBUGS = {
    "mergesort": {
        "buggy": "def mergesort(arr):\n    if len(arr) <= 1: return arr\n    mid = len(arr) // 2\n    left = mergesort(arr[:mid])\n    right = mergesort(arr[mid:])\n    return merge(left, right)",
        "fixed": "def mergesort(arr):\n    if len(arr) <= 1: return arr\n    mid = len(arr) // 2\n    left = mergesort(arr[:mid])\n    right = mergesort(arr[mid:])\n    return list(merge(left, right))",
        "issue": "Fix mergesort - output of merge must be converted to list for consistency."
    },
    "levenshtein": {
        "buggy": "def levenshtein(s, t):\n    if not s or not t: return len(s) or len(t)\n    elif s[0] == t[0]: return levenshtein(s[1:], t[1:])\n    else: return 1 + min(levenshtein(s, t[1:]), levenshtein(s[1:], t), levenshtein(s[1:], t[1:]))",
        "fixed": "def levenshtein(s, t):\n    if not s or not t: return len(s) or len(t)\n    elif s[0] == t[0]: return levenshtein(s[1:], t[1:])\n    else: return 1 + min(levenshtein(s, t[1:]), levenshtein(s[1:], t), levenshtein(s[1:], t[1:]))",
        "issue": "Logic Check: Ensure recursive steps handle edit distance correctly."
    },
    "lis": {
        "buggy": "def lis(arr):\n    ends = {}\n    longest = 0\n    for i, val in enumerate(arr):\n        prefix_lengths = [j for j in range(1, longest + 1) if arr[ends[j]] < val]\n        length = max(prefix_lengths) if prefix_lengths else 0\n        if length == longest or val < arr[ends[length + 1]]:\n            ends[length + 1] = i\n            longest = max(longest, length + 1)\n    return longest",
        "fixed": "def lis(arr):\n    ends = {}\n    longest = 0\n    for i, val in enumerate(arr):\n        prefix_lengths = [j for j in range(1, longest + 1) if arr[ends[j]] < val]\n        length = max(prefix_lengths) if prefix_lengths else 0\n        if length == longest or val < arr[ends[length + 1]]:\n            ends[length + 1] = i\n            longest = max(longest, length + 1)\n    return longest",
        "issue": "Logic: Ensure longest subsequence length tracking is correct."
    },
    "knapsack": {
        "buggy": "def knapsack(capacity, items):\n    memo = {}\n    for i in range(len(items) + 1):\n        for j in range(capacity + 1):\n            if i == 0 or j == 0: memo[i, j] = 0\n            elif items[i-1][0] <= j:\n                memo[i, j] = max(memo[i-1, j], memo[i-1, j - items[i-1][0]] + items[i-1][1])\n            else: memo[i, j] = memo[i-1, j]\n    return memo[len(items), capacity]",
        "fixed": "def knapsack(capacity, items):\n    memo = {}\n    for i in range(len(items) + 1):\n        for j in range(capacity + 1):\n            if i == 0 or j == 0: memo[i, j] = 0\n            elif items[i-1][0] <= j:\n                memo[i, j] = max(memo[i-1, j], memo[i-1, j - items[i-1][0]] + items[i-1][1])\n            else: memo[i, j] = memo[i-1, j]\n    return memo[len(items), capacity]",
        "issue": "Check DP logic for 0/1 knapsack implementation."
    },
    "detect_cycle": {
        "buggy": "def detect_cycle(node):\n    hare = tortoise = node\n    while True:\n        if not hare or not hare.next: return False\n        tortoise = tortoise.next\n        hare = hare.next.next\n        if hare == tortoise: return True",
        "fixed": "def detect_cycle(node):\n    hare = tortoise = node\n    while True:\n        if not hare or not hare.next: return False\n        tortoise = tortoise.next\n        hare = hare.next.next\n        if hare == tortoise: return True",
        "issue": "Verify Floyd's cycle detection logic (Hare and Tortoise)."
    },
    "sqrt": {
        "buggy": "def sqrt(x, epsilon):\n    approx = x / 2\n    while abs(x - approx**2) > epsilon:\n        approx = 0.5 * (approx + x / approx)\n    return approx",
        "fixed": "def sqrt(x, epsilon):\n    approx = x / 2\n    while abs(x - approx**2) > epsilon:\n        approx = 0.5 * (approx + x / approx)\n    return approx",
        "issue": "Newton's method for square root: check loop termination condition."
    },
    "powerset": {
        "buggy": "def powerset(arr):\n    if not arr: return [[]]\n    else:\n        first = arr[0]\n        rest = powerset(arr[1:])\n        return rest + [[first] + subset for subset in rest]",
        "fixed": "def powerset(arr):\n    if not arr: return [[]]\n    else:\n        first = arr[0]\n        rest = powerset(arr[1:])\n        return rest + [[first] + subset for subset in rest]",
        "issue": "Recursive logic: ensure all subsets are generated correctly."
    },
    "breadth_first_search": {
        "buggy": "def breadth_first_search(startnode, goalnode):\n    queue = [startnode]\n    visited = {startnode}\n    while queue:\n        node = queue.pop(0)\n        if node is goalnode: return True\n        for successor in node.successors:\n            if successor not in visited:\n                visited.add(successor)\n                queue.append(successor)\n    return False",
        "fixed": "def breadth_first_search(startnode, goalnode):\n    queue = [startnode]\n    visited = {startnode}\n    while queue:\n        node = queue.pop(0)\n        if node is goalnode: return True\n        for successor in node.successors:\n            if successor not in visited:\n                visited.add(successor)\n                queue.append(successor)\n    return False",
        "issue": "Verify BFS logic: check queue handling and visited set."
    },
    "depth_first_search": {
        "buggy": "def depth_first_search(startnode, goalnode):\n    visited = set()\n    def search(node):\n        if node is goalnode: return True\n        visited.add(node)\n        for successor in node.successors:\n            if successor not in visited:\n                if search(successor): return True\n        return False\n    return search(startnode)",
        "fixed": "def depth_first_search(startnode, goalnode):\n    visited = set()\n    def search(node):\n        if node is goalnode: return True\n        visited.add(node)\n        for successor in node.successors:\n            if successor not in visited:\n                if search(successor): return True\n        return False\n    return search(startnode)",
        "issue": "Verify DFS logic: check recursion and visited state."
    },
    "sieve": {
        "buggy": "def sieve(max_val):\n    primes = []\n    for n in range(2, max_val + 1):\n        if all(n % p > 0 for p in primes): primes.append(n)\n    return primes",
        "fixed": "def sieve(max_val):\n    primes = []\n    for n in range(2, max_val + 1):\n        if all(n % p > 0 for p in primes): primes.append(n)\n    return primes",
        "issue": "Sieve of Eratosthenes: check prime filtering logic."
    }
}

def run_final_benchmark(model, tokenizer):
    model.eval() # Mode for inference
    print("🏆 Testing TestMate on 10 Unseen QuixBugs Algorithms...\n")
    
    for name, data in UNSEEN_QUIXBUGS.items():
        # ChatML format matching your training data
        prompt = f"<|im_start|>system\nRepair algorithmic logic error.<|im_end|>\n<|im_start|>user\nISSUE: {data['issue']}\n\nCODE:\n{data['buggy']}<|im_end|>\n<|im_start|>assistant\n"
        
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=512, 
                temperature=0.1, 
                pad_token_id=tokenizer.pad_token_id
            )
        
        prediction = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

        print(f"📌 Algorithm: {name.upper()}")
        print(f"🤖 TestMate Fix:\n{prediction}")
        print(f"✅ Target Fix:\n{data['fixed']}")
        
        # Check if functionally equivalent
        if prediction.replace(" ", "") == data['fixed'].replace(" ", ""):
            print("\nResult: SUCCESS ✨")
        else:
            print("\nResult: REVIEW ⚠️")
        print("-" * 60)

# Trigger the benchmark using the current memory model
run_final_benchmark(model, tokenizer)

🏆 Testing TestMate on 10 Unseen QuixBugs Algorithms...

📌 Algorithm: MERGESORT
🤖 TestMate Fix:
def mergesort(arr):
    if len(arr) <= 1: return arr
    mid = len(arr) // 2
    left = mergesort(arr[:mid])
    right = mergesort(arr[mid:])
    return list(merge(left, right))
✅ Target Fix:
def mergesort(arr):
    if len(arr) <= 1: return arr
    mid = len(arr) // 2
    left = mergesort(arr[:mid])
    right = mergesort(arr[mid:])
    return list(merge(left, right))

Result: SUCCESS ✨
------------------------------------------------------------
📌 Algorithm: LEVENSHTEIN
🤖 TestMate Fix:
def levenshtein(s, t):
    if not s or not t: return len(s) or len(t)
    elif s[0] == t[0]: return levenshtein(s[1:], t[1:])
    else: return 1 + min(levenshtein(s, t[1:]), levenshtein(s[1:], t), levenshtein(s, t[1:]))
✅ Target Fix:
def levenshtein(s, t):
    if not s or not t: return len(s) or len(t)
    elif s[0] == t[0]: return levenshtein(s[1:], t[1:])
    else: return 1 + min(levenshtein(s, t[1:]), levens

In [14]:
import shutil
from IPython.display import FileLink

shutil.make_archive('testmate_model', 'zip', './qwen-testmate-adapter/final')

FileLink('testmate_model.zip')

/kaggle/working/testmate_model.zip